# Inference Benchmark Report

Generated from the per-run / centralized results DB (SPECIFICATIONS.md §15.1). Latency/SLO math is over the measurement phase only (§12.2).

In [1]:
db_path = '/Users/user/git/inference-benchmarking-tool/experiments/2026-06-15_apertus70b-smoke-dual-platform/20260615-205318_apertus-70b-instruct-2509_vllm_clariden_2164/run_20260615-205318_apertus-70b-instruct-2509_vllm_clariden_2164.db'
run_id = None
out_dir = '/Users/user/git/inference-benchmarking-tool/experiments/2026-06-15_apertus70b-smoke-dual-platform/20260615-205318_apertus-70b-instruct-2509_vllm_clariden_2164'
sessions_per_user_per_hour = {}

In [2]:
from pathlib import Path
import pandas as pd
from tools.reports import analysis, plots
plots.set_style()
out = Path(out_dir); out.mkdir(parents=True, exist_ok=True)
if db_path is None:                       # demo/self-test mode: build the fixture
    from tools.reports.fixtures import build_fixture_db
    db_path = str(out / "fixture_results.db")
    build_fixture_db(db_path)
    run_id = run_id or "fixtureA"
    sessions_per_user_per_hour = sessions_per_user_per_hour or {"chat-short-turns": 3600.0}
report = analysis.load_run(db_path, run_id)
mreq = analysis.measurement_requests(report)
print("loaded run", report.run_id, "—", len(mreq), "measurement-phase requests")


loaded run 20260615-205318_apertus-70b-instruct-2509_vllm_clariden_2164 — 423 measurement-phase requests


## Scenario & assumptions

In [3]:
# Scenario & assumptions (§14.7) — read every chart below in this context.
display(pd.DataFrame(report.manifest.get("mix", [])))
for c in report.manifest.get("classes", []):
    print(f"# {c['name']} — {c.get('summary','')}")
    print("  modelled:", c.get("modelled"))
    print("  NOT modelled:", c.get("not_modelled"))   # must not be missed (§15.1)
    print("  assumptions:", c.get("assumptions"))
print("run assumptions:", report.manifest.get("run_assumptions"))


,expected_request_share,scenario,weight
0,1.0,smoke-synthetic,1.0


# smoke-synthetic — Single-turn synthetic smoke workload for pipeline validation only — results are not findings.
  modelled: ['end-to-end pipeline mechanics only: dataset generation, arrival process, request issuance, metric collection, DB persistence, report rendering', 'unique per-prompt headers so the prefix cache cannot serve synthetic hits']
  NOT modelled: ['NOT a workload model — synthetic filler text carries no semantic realism whatsoever', 'near-zero speculative-decoding acceptance and no meaningful prefix-cache locality (§11.9)', 'no multi-turn structure, no sessions, no think-time', 'results from this scenario must never be used for capacity, Pareto, or procurement claims']
  assumptions: ["input length distribution: lognormal {'mean': 1000.0, 'sigma': 0.5, 'min': 100.0, 'max': 8000.0}", "output length distribution: lognormal {'mean': 200.0, 'sigma': 0.5, 'min': 16.0, 'max': 1024.0}", "turns per session: fixed {'value': 1.0}", 'session mode: open_loop', 'prefix strategy: ap

## Configuration

In [4]:
# Configuration summary
display(pd.DataFrame([{"model": report.model, "backend": report.backend, **report.backend_config}]))


,model,backend,data_parallel_size,disable_custom_all_reduce,enable_prefix_caching,enforce_eager,expert_parallel_size,gpu_memory_utilization,kv_cache_dtype,kv_offloading_backend,kv_offloading_size,max_model_len,max_num_batched_tokens,pipeline_parallel_size,safetensors_load_strategy,speculative_decoding,tensor_parallel_size
0,swiss-ai/Apertus-70B-Instruct-2509,vllm,1,False,True,False,1,0.9,None,None,None,16384,None,1,prefetch,None,4


## System pre-checks

In [5]:
# System pre-checks (§14.6) — warns/fails flagged at the top (§15.1)
sp = report.system_prechecks
if sp.empty:
    print("no system pre-checks recorded")
else:
    flagged = sp[sp["status"].isin(["warn", "fail"])]
    if not flagged.empty:
        print("⚠️  DEGRADED FOUNDATION — interpret all numbers below with care:")
        display(flagged[["metric", "measured", "expected", "status"]])
    display(sp[["metric", "measured", "expected", "tolerance_pct", "status"]])


,metric,measured,expected,tolerance_pct,status
0,nccl_all_gather_128_mib,284.8200,283.500,-10.0,pass
1,nccl_all_reduce_128_mib,317.3700,317.700,-10.0,pass
2,nccl_alltoall_128_mib,307.7800,306.200,-10.0,pass
3,nvshmem_alltoall_latency_128_kib,12.5984,NaN,NaN,pass
4,sequential_read_1_mib_blocks,0.2350,0.063,-15.0,pass
5,parallel_read_aggregate,1.0299,NaN,NaN,pass
6,buffered_read_aggregate,16.4248,NaN,NaN,pass


## Model loading times

In [6]:
# Model loading times (§10.2), per instance
cols = ["instance_id", "node", "model_load_total_s", "model_load_weights_s",
        "model_load_engine_init_s", "model_load_cuda_graph_capture_s", "model_load_inductor_compile_s"]
display(report.instances[[c for c in cols if c in report.instances.columns]])


,instance_id,node,model_load_total_s,model_load_weights_s,model_load_engine_init_s,model_load_cuda_graph_capture_s,model_load_inductor_compile_s
0,i0,nid007145,455.103263,128.078047,144.22,14.0,118.33


## Latency vs λ

In [7]:
# TTFT vs λ with the per-class SLO line (§15.1)
ttft_slo = next((s["threshold"] for s in report.slos if s["metric"] == "ttft_ms"), None)
fig = plots.latency_figure(report, "ttft_ms", slo_threshold=ttft_slo)
fig.savefig(out / "ttft.png", bbox_inches="tight"); fig


<Figure size 770x550 with 2 Axes>

In [8]:
# Inter-token latency (TPOT/ITL) vs λ
tpot_slo = next((s["threshold"] for s in report.slos if s["metric"] == "tpot_ms"), None)
fig = plots.latency_figure(report, "tpot_ms", slo_threshold=tpot_slo)
fig.savefig(out / "itl.png", bbox_inches="tight"); fig


<Figure size 770x550 with 2 Axes>

In [9]:
# Per-class breakdown (mixed runs, §11.4/§15.1): TTFT percentiles by scenario
display(analysis.latency_vs_lambda(mreq, "ttft_ms", by_scenario=True))
display(analysis.failure_rate_vs_lambda(mreq, by_scenario=True))


,rate_lambda,scenario,p50,p90,p95,p99,n
0,0.5,smoke-synthetic,104.687224,151.603824,173.850644,203.926050,73
1,1.0,smoke-synthetic,108.825388,174.906187,202.249423,251.891240,104
2,2.0,smoke-synthetic,115.164103,198.659774,248.694188,293.121525,246


,rate_lambda,scenario,error_rate_pct,n
0,0.5,smoke-synthetic,0.0,73
1,1.0,smoke-synthetic,0.0,104
2,2.0,smoke-synthetic,0.0,246


## SLO attainment & λ*

In [10]:
# SLO attainment per λ and the derived λ* (§13.4)
att = analysis.evaluate_slos(report)
lam_star = analysis.lambda_star(report)
print("λ* =", lam_star)
display(att)


λ* = 2.0


,rate_lambda,scenario,metric,percentile,threshold,measured,passed
0,0.5,smoke-synthetic,ttft_ms,p95,5000.0,173.850644,True
1,0.5,all,error_rate_pct,NaN,5.0,0.000000,True
2,1.0,smoke-synthetic,ttft_ms,p95,5000.0,202.249423,True
3,1.0,all,error_rate_pct,NaN,5.0,0.000000,True
4,2.0,smoke-synthetic,ttft_ms,p95,5000.0,248.694188,True
5,2.0,all,error_rate_pct,NaN,5.0,0.000000,True


## Supportable users

In [11]:
# Supportable-users estimate at λ* (§15.1) — edit sessions_per_user_per_hour above
users = analysis.supportable_users(report, lam_star, sessions_per_user_per_hour)
display(users if not users.empty else "λ* undefined — extend the sweep toward lower rates")


,scenario,sessions_started,session_throughput_per_s,mean_session_wall_s,sessions_per_user_per_hour,supportable_users,concurrent_sessions
0,smoke-synthetic,246,2.05,3.469472,None,None,7.112417


## Response quality

In [12]:
# Response quality (§13.5): Stage-A gate, Stage-B scores, capacity-vs-quality
q = analysis.quality_summary(report)
if q["quality_flagged"]:
    print("⚠️  QUALITY-FLAGGED: Stage-A gate failed under on_fail=continue (§15.1)")
display(q["compare"][["suite", "eval_concurrency", "score"]] if not q["compare"].empty
        else "no Stage-B quality rows")
runs = analysis.list_runs(db_path)
if len(runs) > 1:
    print("Capacity vs quality across deployment configs:")
    display(analysis.capacity_vs_quality(db_path, runs, sessions_per_user_per_hour))


'no Stage-B quality rows'

## Hardware utilisation

In [13]:
# Hardware utilisation vs λ — untapped headroom (§13.3/§15.1)
fig = plots.hardware_figure(report, ["gpu_sm_active_pct", "gpu_tensor_active_pct"])
if fig is not None:
    fig.savefig(out / "hardware.png", bbox_inches="tight")
fig if fig is not None else "no hardware telemetry recorded"


'no hardware telemetry recorded'

## Raw data

In [14]:
# Raw per-rate-level table
raw = analysis.latency_vs_lambda(mreq, "ttft_ms").merge(
    analysis.failure_rate_vs_lambda(mreq), on="rate_lambda", how="outer", suffixes=("_ttft", "")
)
display(raw)


,rate_lambda,p50,p90,p95,p99,n_ttft,error_rate_pct,n
0,0.5,104.687224,151.603824,173.850644,203.926050,73,0.0,73
1,1.0,108.825388,174.906187,202.249423,251.891240,104,0.0,104
2,2.0,115.164103,198.659774,248.694188,293.121525,246,0.0,246
